# Sample Description Table — `data3.xlsx`

Builds a publication-style **sample description table** and exports it as a
formatted Excel file to `D:\Headway\Tables`.

Table structure (columns):
1. **Variable** — organised in large groups, then individual variables
2. **Sample size, n (%)** — count and distribution
3. **Range / Class** — min–max for continuous, class label for categorical
4. **Mean ± SD** — continuous variables only

Run the cells in order.

### 1. Imports and paths

In [1]:
import os
import numpy as np
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

# ---- Paths -----------------------------------------------------------
BASE_DIR   = r"D:\Headway"
DATA_PATH  = os.path.join(BASE_DIR, "data3.xlsx")
TABLES_DIR = os.path.join(BASE_DIR, "Tables")
os.makedirs(TABLES_DIR, exist_ok=True)

OUT_PATH = os.path.join(TABLES_DIR, "Table_Sample_Description.xlsx")
print("Data file :", DATA_PATH)
print("Output    :", OUT_PATH)

Data file : D:\Headway\data3.xlsx
Output    : D:\Headway\Tables\Table_Sample_Description.xlsx


### 2. Load data

In [2]:
df = pd.read_excel(DATA_PATH)
N = len(df)
print(f"Loaded {N} observations, {df.shape[1]} columns")
print("Columns:", list(df.columns))
df.head()

Loaded 898 observations, 11 columns
Columns: ['V_Target', 'V_Leading_Class', 'Pair', 'Time_Headway', 'Target_Speed_km/hr', 'Leading_Speed_km/hr', 'Speed_Difference', 'Off_centeredness', 'Occupancy', 'Flow_pcu/hr', 'Site']


,V_Target,V_Leading_Class,Pair,Time_Headway,Target_Speed_km/hr,Leading_Speed_km/hr,Speed_Difference,Off_centeredness,Occupancy,Flow_pcu/hr,Site
0,BTW,MT_2W,BTW_following_MT_2W,0.733333,24.300000,30.375000,-6.075000,True,True,2355.000000,Shahjahanpur
1,BTW,4W,BTW_following_4W,2.866667,22.090909,34.714286,-12.623377,True,True,2241.000000,Shahjahanpur
2,BTW,MT_2W,BTW_following_MT_2W,3.233333,16.200000,30.375000,-14.175000,True,True,2382.000000,Shahjahanpur
3,BTW,4W,BTW_following_4W,2.366667,27.000000,32.400000,-5.400000,False,True,2292.000000,Shahjahanpur
4,BTW,4W,BTW_following_4W,3.333333,27.000000,32.400000,-5.400000,True,True,2005.217031,Shahjahanpur


### 3. Configuration — variable groups, labels, and units

Edit `LABELS`, `UNITS`, and `CLASS_LABELS` if you want different display text.
`CATEGORICAL_GROUPS` and `CONTINUOUS_GROUP` define how variables are organised
in the table. Categorical classes are ordered by descending frequency.

In [3]:
# Human-readable variable labels
LABELS = {
    "V_Target":            "Target vehicle type",
    "V_Leading_Class":     "Leading vehicle class",
    "Pair":                "Following pair (target following leader)",
    "Site":                "Study site",
    "Off_centeredness":    "Off-centeredness",
    "Occupancy":           "Occupancy",
    "Time_Headway":        "Time headway",
    "Target_Speed_km/hr":  "Target vehicle speed",
    "Leading_Speed_km/hr": "Leading vehicle speed",
    "Speed_Difference":    "Speed difference (target - leader)",
    "Flow_pcu/hr":         "Traffic flow",
}

# Units for continuous variables (appended to the label)
UNITS = {
    "Time_Headway":        "s",
    "Target_Speed_km/hr":  "km/h",
    "Leading_Speed_km/hr": "km/h",
    "Speed_Difference":    "km/h",
    "Flow_pcu/hr":         "pcu/h",
}

# Optional relabelling of individual class values (leave empty to keep raw)
CLASS_LABELS = {
    "Off_centeredness": {True: "Off-centered", False: "Centered"},
    "Occupancy":        {True: "Occupied",     False: "Not occupied"},
}

# --- Grouping structure ----------------------------------------------
# Each tuple: (group heading, [list of column names])
CATEGORICAL_GROUPS = [
    ("Vehicle classification", ["V_Target", "V_Leading_Class", "Pair"]),
    ("Study context",          ["Site"]),
    ("Following configuration", ["Off_centeredness", "Occupancy"]),
]

CONTINUOUS_GROUP = (
    "Kinematic and traffic variables",
    ["Time_Headway", "Target_Speed_km/hr", "Leading_Speed_km/hr",
     "Speed_Difference", "Flow_pcu/hr"],
)

### 4. Build the table rows

Row types produced:
- `group`  — bold group heading spanning the table
- `varcat` — a categorical variable name (sub-header)
- `class`  — one class/level of a categorical variable (n, %, class label)
- `cont`   — a continuous variable (n, %, range, mean ± SD)

In [4]:
DEC = 2   # decimal places for continuous statistics

def fmt_num(x, dec=DEC):
    return f"{x:.{dec}f}"

rows = []  # list of dicts: {variable, n_pct, range_class, mean_sd, rtype}

# ---------- Categorical groups ----------
for group_name, cols in CATEGORICAL_GROUPS:
    rows.append({"variable": group_name, "n_pct": "", "range_class": "",
                 "mean_sd": "", "rtype": "group"})
    for col in cols:
        rows.append({"variable": LABELS.get(col, col), "n_pct": "",
                     "range_class": "", "mean_sd": "", "rtype": "varcat"})
        vc = df[col].value_counts(dropna=False)          # descending freq
        cmap = CLASS_LABELS.get(col, {})
        for level, cnt in vc.items():
            disp = cmap.get(level, level)
            rows.append({
                "variable":    "",
                "n_pct":       f"{cnt} ({100*cnt/N:.1f}%)",
                "range_class": str(disp),
                "mean_sd":     "",
                "rtype":       "class",
            })

# ---------- Continuous group ----------
group_name, cols = CONTINUOUS_GROUP
rows.append({"variable": group_name, "n_pct": "", "range_class": "",
             "mean_sd": "", "rtype": "group"})
for col in cols:
    s = df[col].astype(float)
    unit = UNITS.get(col, "")
    label = LABELS.get(col, col) + (f" ({unit})" if unit else "")
    rng = f"{fmt_num(s.min())} \u2013 {fmt_num(s.max())}"       # en-dash
    msd = f"{fmt_num(s.mean())} \u00b1 {fmt_num(s.std())}"       # mean ± sd
    n_non = int(s.notna().sum())
    rows.append({
        "variable":    label,
        "n_pct":       f"{n_non} ({100*n_non/N:.1f}%)",
        "range_class": rng,
        "mean_sd":     msd,
        "rtype":       "cont",
    })

table_df = pd.DataFrame(rows)
print(f"{len(table_df)} rows built")
table_df[["variable", "n_pct", "range_class", "mean_sd"]].head(20)

38 rows built


,variable,n_pct,range_class,mean_sd
0,Vehicle classification,,,
1,Target vehicle type,,,
2,,669 (74.5%),BTW,
3,,229 (25.5%),PR,
4,Leading vehicle class,,,
5,,293 (32.6%),4W,
6,,290 (32.3%),MT_3W,
7,,168 (18.7%),NMT_3W,
8,,96 (10.7%),MT_2W,
9,,51 (5.7%),NMT_2W,


### 5. Write to Excel and apply formatting

In [5]:
COLS = ["Variable", "Sample size, n (%)", "Range / Class", "Mean \u00b1 SD"]
KEYS = ["variable", "n_pct", "range_class", "mean_sd"]

# Write plain values first with pandas, then format with openpyxl
out_df = table_df[KEYS].copy()
out_df.columns = COLS

title = "Table X. Description of the analysis sample"
subtitle = f"(N = {N} following pairs)"

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as xw:
    out_df.to_excel(xw, index=False, sheet_name="Sample Description", startrow=2)

wb = load_workbook(OUT_PATH)
ws = wb["Sample Description"]
ncol = len(COLS)
last_col = chr(ord("A") + ncol - 1)

# ---- Fonts / styles ----
FONT = "Times New Roman"
thin  = Side(style="thin",  color="000000")
med   = Side(style="medium", color="000000")
hdr_fill  = PatternFill("solid", fgColor="D9E1F2")
grp_fill  = PatternFill("solid", fgColor="F2F2F2")

# ---- Title rows ----
ws.merge_cells(f"A1:{last_col}1")
ws["A1"] = title
ws["A1"].font = Font(name=FONT, size=12, bold=True)
ws["A1"].alignment = Alignment(horizontal="left")

ws.merge_cells(f"A2:{last_col}2")
ws["A2"] = subtitle
ws["A2"].font = Font(name=FONT, size=10, italic=True)
ws["A2"].alignment = Alignment(horizontal="left")

# ---- Header row (row 3) ----
hdr_row = 3
for j, name in enumerate(COLS):
    c = ws.cell(row=hdr_row, column=j+1, value=name)
    c.font = Font(name=FONT, size=10, bold=True)
    c.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    c.fill = hdr_fill
    c.border = Border(top=med, bottom=med)

# ---- Body rows ----
first_body = hdr_row + 1
for i, (_, r) in enumerate(table_df.iterrows()):
    excel_row = first_body + i
    rtype = r["rtype"]
    for j in range(ncol):
        c = ws.cell(row=excel_row, column=j+1)
        c.font = Font(name=FONT, size=10)
        c.alignment = Alignment(horizontal="left" if j == 0 else "center",
                                vertical="center", wrap_text=(j == 0))
    a = ws.cell(row=excel_row, column=1)
    if rtype == "group":
        ws.merge_cells(start_row=excel_row, start_column=1,
                       end_row=excel_row, end_column=ncol)
        a.font = Font(name=FONT, size=10, bold=True)
        a.fill = grp_fill
    elif rtype == "varcat":
        a.font = Font(name=FONT, size=10, bold=True, italic=True)
    elif rtype == "class":
        a.value = ""  # class label lives in Range/Class column
        ws.cell(row=excel_row, column=1).alignment = Alignment(indent=1)
    elif rtype == "cont":
        a.font = Font(name=FONT, size=10)

# ---- Bottom border on last row ----
last_row = first_body + len(table_df) - 1
for j in range(ncol):
    ws.cell(row=last_row, column=j+1).border = Border(bottom=med)

# ---- Column widths ----
widths = [40, 20, 18, 18]
for j, w in enumerate(widths):
    ws.column_dimensions[chr(ord("A") + j)].width = w

# ---- Footnote ----
note_row = last_row + 2
ws.merge_cells(start_row=note_row, start_column=1, end_row=note_row, end_column=ncol)
note = ws.cell(row=note_row, column=1,
    value=("Note. Percentages for categorical variables are of the total sample. "
           "For continuous variables, Range is min\u2013max and values are Mean \u00b1 SD. "
           "SD = standard deviation."))
note.font = Font(name=FONT, size=8, italic=True)
note.alignment = Alignment(horizontal="left", wrap_text=True)

wb.save(OUT_PATH)
print("Saved:", OUT_PATH)

Saved: D:\Headway\Tables\Table_Sample_Description.xlsx


### 6. Quick preview of the exported table

In [6]:
preview = pd.read_excel(OUT_PATH, sheet_name="Sample Description", header=2)
preview.fillna("").head(40)

,Variable,"Sample size, n (%)",Range / Class,Mean ± SD
0,Vehicle classification,,,
1,Target vehicle type,,,
2,,669 (74.5%),BTW,
3,,229 (25.5%),PR,
4,Leading vehicle class,,,
5,,293 (32.6%),4W,
6,,290 (32.3%),MT_3W,
7,,168 (18.7%),NMT_3W,
8,,96 (10.7%),MT_2W,
9,,51 (5.7%),NMT_2W,
